<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Select and filter

**[C04-M02-L01 · Select and filter](https://learning.nextia-ai.com/courses/sql/m02/select-and-filter/)** · C04, SQL and Data Preparation for AI · Module 2, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** choose the columns and rows that you need with `SELECT`, `WHERE`, `ORDER BY` and `LIMIT`, and find the two quiet mistakes of filters on real ticket data: NULL values, and categories with more than one spelling.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). |
| **You should know** | The three tables, `customers`, `tickets` and `outcomes`, and their grain, from [Dataset anatomy](https://learning.nextia-ai.com/courses/sql/m01/dataset-anatomy/). That `tickets` has 1,417 rows but 1,403 distinct tickets, from [Relational concepts](https://learning.nextia-ai.com/courses/sql/m01/relational-concepts/). |
| **You do not need** | The local project from Module 1. The setup below builds the same database here. |
| **Tested** | 2026-10-07, Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m02/select-and-filter/) has the full explanations, the tables of examples and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your query between the triple quotes `"""`, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M02-L01-select-and-filter/select-and-filter-solution.ipynb).

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A query on six rows

Larkfield is a fictional online shop for home and garden products. Each contact with its support team is a ticket. Mei, a data engineer, answers the support team's questions with SQL. The data is made up for this course; no real people are in it.

A **query** is a question that you send to a database, written in SQL. The database answers with a table of rows. Start with a table small enough to see completely: six tickets.

| ticket_id | channel | team | priority |
|---|---|---|---|
| T-1 | email | delivery | 1 |
| T-2 | chat | returns | 3 |
| T-3 | Email | delivery | 2 |
| T-4 | NULL | payment | 1 |
| T-5 | phone | delivery | 3 |
| T-6 | email | account | 2 |

**NULL** means "no value is stored here". For T-4, nobody recorded the channel. NULL is not the text `'NULL'` and not an empty text `''`.

The next cell makes this table in memory, as `tiny`. Run the cell. It prints nothing.

In [ ]:
tiny = sqlite3.connect(":memory:")
tiny.executescript("""
CREATE TABLE tickets (ticket_id TEXT, channel TEXT, team TEXT, priority INTEGER);
INSERT INTO tickets VALUES
    ('T-1', 'email', 'delivery', 1),
    ('T-2', 'chat',  'returns',  3),
    ('T-3', 'Email', 'delivery', 2),
    ('T-4', NULL,    'payment',  1),
    ('T-5', 'phone', 'delivery', 3),
    ('T-6', 'email', 'account',  2);
""");

A query has parts called **clauses**. `SELECT` says which columns to show, `FROM` which table to read, and `WHERE` which rows to keep. The database reads the table, keeps the rows that pass the `WHERE` test, then picks the columns.

> **Predict.** How many rows does this query return, and which tickets? Write it down, then run the cell.

In [ ]:
sql("""
SELECT ticket_id, channel
FROM tickets
WHERE priority = 1;
""", db=tiny)

> **Check.** You should see **2 rows**: T-1 and T-4. The channel of T-4 is NULL.

> **Your turn: change one thing.** In the cell above, change `priority = 1` to `team = 'delivery'`, and run it again. You should see 3 rows: T-1, T-3 and T-5.

## 2. What WHERE keeps

`WHERE` tests each row. It keeps a row only when the test is **true**. Here are two "opposite" tests on the six tickets.

> **Predict.** Six rows go in. How many rows do the two tests keep together? Then run the cell.

In [ ]:
sql("""
SELECT
    (SELECT COUNT(*) FROM tickets WHERE channel = 'email')  AS is_email,
    (SELECT COUNT(*) FROM tickets WHERE channel <> 'email') AS not_email;
""", db=tiny)

> **Check.** You should see `2` and `3`. 2 + 3 = 5, not 6. Two things are surprises:

- **T-3 is not an email ticket** for `channel = 'email'`. Text comparison is exact: `'Email'` with a capital E is a different value.
- **T-4 is in neither result.** Any comparison with NULL, such as `NULL = 'email'` or `NULL <> 'email'`, gives **unknown**, not true or false. `WHERE` keeps only true, so the row disappears from both. You get no error and no warning.

To test for NULL, use `IS NULL` or `IS NOT NULL`. The next query reads no table; it only calculates three tests. SQLite shows true as 1 and false as 0.

> **Predict.** Which of the three tests gives a real answer? Then run the cell.

In [ ]:
sql("""
SELECT
    NULL = NULL      AS equals_null,
    NULL <> 'email'  AS not_email,
    NULL IS NULL     AS is_null;
""", db=tiny)

> **Check.** You should see `NULL`, `NULL` and `1`. Only `IS NULL` gives a real answer.

## 3. The tools of a filter

From here on, the queries run on the real `tickets` table, which has 1,417 rows. A filter can use `=` and `<>` (equal, not equal), `<`, `<=`, `>` and `>=`, `AND`, `OR`, `NOT`, `IN (…)` (one of a list of values), `LIKE` (a pattern: `%` is any text, `_` is one character) and `IS NULL`. The [lesson page](https://learning.nextia-ai.com/courses/sql/m02/select-and-filter/#the-tools-of-a-filter) has a table with an example and a row count for each.

Text values go in single quotes: `'payment'`. Numbers do not: `priority = 1`.

SQL does `AND` before `OR`, as arithmetic does × before +. So `team = 'payment' OR team = 'returns' AND priority = 1` means "all payment tickets, plus urgent returns tickets".

> **Predict.** Which of the three counts below are the same? Then run the cell.

In [ ]:
sql("""
SELECT
    (SELECT COUNT(*) FROM tickets
       WHERE team = 'payment' OR team = 'returns' AND priority = 1) AS no_brackets,
    (SELECT COUNT(*) FROM tickets
       WHERE (team = 'payment' OR team = 'returns') AND priority = 1) AS with_brackets,
    (SELECT COUNT(*) FROM tickets
       WHERE team IN ('payment', 'returns') AND priority = 1) AS with_in;
""")

> **Check.** You should see `280`, `98` and `98`. If you mean "urgent tickets of the two teams", write brackets, or use `IN`, which is easier to read.

## 4. Worked example: how many email tickets?

Grace, the support lead, asks Mei: "How many tickets came in by email?" Mei writes the obvious query first. `COUNT(*)` counts rows; the next lesson explains it in detail. Run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS email_rows
FROM tickets
WHERE channel = 'email';
""")

> **Check.** You should see **541**.

Before she sends it, Mei checks which values the column really contains. `SELECT DISTINCT` returns each different value once. On your computer, this is the file `queries/21-channel-values.sql`.

> **Predict.** The shop has 5 real channels: email, chat, web form, phone, and "not recorded". How many distinct values do you expect? Then run the cell.

In [ ]:
sql("""
-- Every spelling of channel in the tickets table.
SELECT DISTINCT channel
FROM tickets
ORDER BY channel;
""")

> **Check.** You should see **9 rows**, and the first is NULL. ` chat` has a space at the start. `Email` and `EMAIL` are email with capitals. `webform` is `web_form` without the underscore. This is a **messy category**: one meaning, several spellings. SQLite sorts NULL first, then by character code, so a space comes before capitals, and capitals come before small letters.

For a report, Mei compares a standard form of the text. `TRIM` removes spaces at the start and the end, and `LOWER` changes capitals to small letters. Run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS email_rows
FROM tickets
WHERE LOWER(TRIM(channel)) = 'email';
""")

> **Check.** You should see **568**: 541 + 20 (`Email`) + 7 (`EMAIL`). The first query missed 27 rows, about 5% of the email work.

Grace asks a second question: "And how many came in by another channel?" Mei predicts 1,417 − 541 = 876, and checks with three counts.

> **Predict.** Do the three counts add up to 1,417? Then run the cell.

In [ ]:
sql("""
SELECT
    (SELECT COUNT(*) FROM tickets WHERE channel = 'email')  AS is_email,
    (SELECT COUNT(*) FROM tickets WHERE channel <> 'email') AS not_email,
    (SELECT COUNT(*) FROM tickets WHERE channel IS NULL)    AS no_channel;
""")

> **Check.** You should see `541`, `838` and `38`. The two "opposite" filters cover only 1,379 rows. The 38 rows with no channel are in neither, exactly as T-4 was. 541 + 838 + 38 = 1,417.

These numbers count **rows**, not tickets: the table has 14 duplicate rows. And Mei does not change the data: `LOWER(TRIM(channel))` changes only what this one query compares. Module 4 cleans the column itself.

## 5. Columns, aliases and sorting

In a saved query, name the columns that you need instead of `SELECT *`. The query then says what it uses, and it fails clearly when a column is renamed. An **alias** gives a column in the result a new name, with `AS`.

`ORDER BY` sorts the rows: `ASC` (the default) from small to large, `DESC` from large to small. Without `ORDER BY`, the order of the rows is not defined, so `LIMIT` without `ORDER BY` gives "some 10 rows". Dates in this database are text in the form `YYYY-MM-DD HH:MM:SS`, which sorts in time order.

Grace wants to read the newest urgent delivery tickets. On your computer, this is the file `queries/22-urgent-delivery.sql`.

> **Predict.** Is the first row from June 2026? Then run the cell.

In [ ]:
sql("""
SELECT
    ticket_id,
    created_at,
    LOWER(TRIM(channel)) AS channel_for_report,
    word_count AS words
FROM tickets
WHERE priority = 1
  AND team = 'delivery'
ORDER BY created_at DESC
LIMIT 10;
""")

> **Check.** You should see **10 rows**. The first is T-11384, created on 2026-06-29 11:40:07, by chat, with 84 words.

> **Your turn: change one thing.** In the cell above, remove the two `WHERE` lines (`WHERE priority = 1` and `AND team = 'delivery'`), and run it again. The first row is now T-10147, created in **2036**. Sorting shows the extremes of a column, so it is a quick way to find impossible values.

## 6. Your turn: urgent tickets with no channel

> **Your turn.** Grace wants to know which urgent tickets have no recorded channel. The query below has a mistake, so it returns no rows. Find the mistake and fix it. Then run the check cell. When it is correct, you see 5 rows, the first is T-10133, and the check cell prints `Check passed.`

In [ ]:
no_channel = """
SELECT ticket_id, channel, team
FROM tickets
WHERE priority = 1
  AND channel IS NULL
ORDER BY ticket_id;
"""
sql(no_channel)

In [ ]:
check(no_channel, rows=5, columns=3, expected="2c6986ddac855dfb", ordered=True)

## 7. Independent variation: everything except chat

Omar, the product manager, asks: "How many tickets did **not** come in by chat? Include the tickets with no recorded channel, and count ` chat` with a space as chat."

> **Your turn.** The starting query gives a number, but it is wrong in two ways. Change only the `WHERE` clause. Then run the check cell.

> **Predict.** The table has 1,417 rows, and 421 of them are chat in some spelling. What is the right answer? Write it down before you run your query.

In [ ]:
not_chat = """
SELECT COUNT(*) AS not_chat_rows
FROM tickets
WHERE LOWER(TRIM(channel)) <> 'chat'
   OR channel IS NULL;
"""
sql(not_chat)

In [ ]:
check(not_chat, rows=1, columns=1, expected="ddd6fb86043ae6d4")

Always check a "not" filter: the rows that pass plus the rows that fail must equal the whole table. Run the cell. It adds your count to the chat rows and compares the sum with the table.

In [ ]:
columns, rows = run(not_chat)
chat = con.execute("SELECT COUNT(*) FROM tickets WHERE LOWER(TRIM(channel)) = 'chat'").fetchone()[0]
expect("not chat + chat", rows[0][0] + chat, 1417)

> **Check.** With the starting query, you see `Not yet: not chat + chat is 1387; expected 1417.` With the correct query, you see `Check passed: not chat + chat is 1417.`

## 8. Failure case: a "not email" filter loses the rows with no channel

Tomás builds a training table of "all tickets that did not come by email". He uses `WHERE channel <> 'email'` and gets 838 rows. But 1,417 − 541 = 876. Where are the other 38?

For the 38 rows where `channel` is NULL, `channel <> 'email'` is unknown, so `WHERE` drops them. The same filter also keeps the 27 rows of `Email` and `EMAIL`, which are email tickets.

**Diagnose:** count both sides of the filter and the NULLs, and add them up, as in section 4. **Fix:** decide what the NULL rows mean, and write it in the query. Here, "no channel" is not email.

> **Predict.** How many rows does the fixed filter keep? Then run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS not_email_rows
FROM tickets
WHERE LOWER(TRIM(channel)) <> 'email'
   OR channel IS NULL;
""")

> **Check.** You should see **849** = 1,417 − 568.

**In AI work:** with Tomás's filter, the model never sees a ticket with no channel during training. In production, such tickets still arrive, and the model's score on them is unknown. A filter that silently removes a group is one way that training data stops matching real data.

## Check your understanding and recap

Answer the three **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/sql/m02/select-and-filter/). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that `WHERE` keeps only the rows where a test is true, so a comparison with NULL drops the row; that text comparison is exact, so you look at the distinct values of a category before you filter on it; and that `LIMIT` needs `ORDER BY`. You check a filter by counting: the rows that pass, the rows that fail and the NULL rows must add up to the table.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Aggregate correctly](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m02/select-and-filter/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.